# Coachly NLU — Fine Tuning
**Prima di tutto:** `Runtime → Change runtime type → T4 GPU`

Esegui le celle in ordine con `Shift+Enter`.

In [ ]:
# Cella 1 — Monta Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cella 2 — Copia il progetto da Drive
import os, shutil

DRIVE_SRC = '/content/drive/MyDrive/voice-ml-recognizer'
LOCAL_DIR = '/content/proj'

if os.path.exists(LOCAL_DIR):
    shutil.rmtree(LOCAL_DIR)
shutil.copytree(DRIVE_SRC, LOCAL_DIR)
os.chdir(LOCAL_DIR)
print('Working dir:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
# Cella 3 — Installa dipendenze e verifica
import subprocess, sys

pkgs = [
    'transformers>=4.38.0',
    'seqeval>=1.2.2',
    'onnx>=1.15.0',
    'onnxruntime>=1.17.0',
    'pytorch-crf>=0.7.2',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
print('Installazione completata.\n')

# Verifica import — se uno fallisce vedi subito quale
import torch;          print(f'torch        {torch.__version__}')
from torchcrf import CRF; print('torchcrf     ok')
import transformers;   print(f'transformers {transformers.__version__}')
import seqeval;        print(f'seqeval      {seqeval.__version__}')
import onnx;           print(f'onnx         {onnx.__version__}')
import onnxruntime;    print(f'onnxruntime  {onnxruntime.__version__}')
print()
print(f'GPU: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# Cella 4 — Genera dataset (skip se già presente)
import os, json, subprocess, sys

if os.path.exists('data/train.json'):
    with open('data/train.json') as f:
        n = len(json.load(f))
    print(f'Dataset già presente ({n} esempi train) — skip.')
else:
    print('Generazione dataset...')
    subprocess.run([sys.executable, 'generate-dataset.py'], check=True)

In [ ]:
# Cella 5 — Fine tuning
# L'output appare in tempo reale E viene salvato in output/workout_nlu/training.log
# Se la sessione Colab crasha, riesegui da cella 1: riparte dall'epoch salvata su Drive.
!python colab_train.py

In [ ]:
# Cella 6 — Mostra log completo (utile se la cella 5 era già terminata)
log_path = '/content/proj/output/workout_nlu/training.log'
if os.path.exists(log_path):
    with open(log_path) as f:
        print(f.read())
else:
    print('Log non trovato — il training non è ancora partito.')

In [ ]:
# Cella 7 — Verifica risultati finali
import json, os

results_path = '/content/proj/output/workout_nlu/results.json'
if os.path.exists(results_path):
    with open(results_path) as f:
        r = json.load(f)
    print('=== Risultati finali ===')
    print(f"Best val combined : {r['best_val']:.4f}")
    print(f"Test intent_acc   : {r['test']['intent_acc']:.4f}")
    print(f"Test slot_f1      : {r['test']['slot_f1']:.4f}")
    print(f"Test combined     : {r['test']['combined']:.4f}")
else:
    print('results.json non trovato — training non completato.')

out_dir = '/content/proj/output/workout_nlu'
if os.path.exists(out_dir):
    print('\nFile prodotti:')
    for fname in sorted(os.listdir(out_dir)):
        size = os.path.getsize(os.path.join(out_dir, fname))
        print(f'  {fname:35s} {size/1e6:6.1f} MB')

In [ ]:
# Cella 8 — Salva tutto su Drive
import os, shutil

DRIVE_OUT = '/content/drive/MyDrive/coachly_nlu'
os.makedirs(DRIVE_OUT, exist_ok=True)
shutil.copytree('/content/proj/output/workout_nlu', DRIVE_OUT, dirs_exist_ok=True)
print('Salvato su Drive:', DRIVE_OUT)
print('File:', os.listdir(DRIVE_OUT))